<a href="https://colab.research.google.com/github/nika19du/AI-for-Developers-summer-2026-/blob/main/HITL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.0/570.0 kB 23.0 MB/s eta 0:00:00


In [13]:
import json
from IPython.display import HTML
from google.colab import userdata
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.messages import HumanMessage
from langchain.tools import tool
from langchain_core.messages import BaseMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command, Interrupt
from pydantic import SecretStr
from typing import List

openai_api_key = SecretStr(userdata.get('OPENAI_API_KEY'))

def print_conversation(conversation: List[BaseMessage]):
    for message in conversation:
        message.pretty_print()


def print_interrupts(interrupts: List[Interrupt]):
    if not interrupts:
        print("There are no interrupts")
        return

    for interrupt in interrupts:
        for action_request in interrupt.value["action_requests"]:
            display(HTML(f'<div style="border: 1px dashed red; margin: 5px; padding: 10px; white-space: pre-wrap;">{action_request["description"]}</div>'))

In [7]:
@tool
def search_travel_options(destination: str) -> str:
  """
  Call this tool to search for flight and accommodation options for a trip.
  """
  options = {
      "destination": destination,
      "flights": [
          {"flight_code": "LH170", "departure": "08:10", "price_eur": 189},
          {"flight_code": "FR402", "departure": "09:05", "price_eur": 129},
      ],
      "hotels": [
          { "hotel_name": "Asd Hub Hotel", "price_eur": 156 },
          { "hotel_name": "Spring View Stay", "price_eur": 144 }
      ]
  }

  return json.dumps(options, indent = 2)


@tool
def book_flight(traveler_name: str, flight_code: str) -> str:
  """
  Call this tool to book a flight.
  """
  return f"Booked flight {flight_code} for {traveler_name}"


@tool
def book_hotel(traveler_name: str, hotel_name: str, nights: int) -> str:
  """
  Call this tool to book a hotel.
  """
  return f"Booked {nights} night(s) at {hotel_name} for {traveler_name}"

за да използваме human-in-the-loop трябва да сетнем short-term-memory тъй като всеки един interrupt форсира прекратяването на работния цикъл на агента с което приключва текущия рън и когато предоставим approve, reject, edit, - тогава възобновяваме изпълнението, възобновяваме интеракцията с агнта само че вече в отделен рън, който има нужа от историята, какво се е случило.

In [15]:
agent = create_agent(
    model = ChatOpenAI(model="gpt-5-nano", api_key = openai_api_key, reasoning_effort = "low"),
    tools = [search_travel_options, book_flight, book_hotel],
    checkpointer = InMemorySaver(),
    middleware= [
        HumanInTheLoopMiddleware(
            interrupt_on ={
                book_flight.name: True,
                book_hotel.name: True, # за кои инструменти трябва да следи
            }
        )
    ]
)

In [22]:
result_1 = agent.invoke(
    input = {
        "messages": [HumanMessage("Plan a one-night business trip to Berlin for Dana. Search options, book a morning flight, and reserve one hotel night for up to 150 EUR. If there are many possible options, take the initiative and select the most affordable one.")],
    },
    config = {
        "configurable": {
            "thread_id": "berlin_trip_3"
        }
    }
)

In [24]:
print_conversation(result_1['messages'])
print_interrupts(result_1.get("__interrupt__", [])) # търсим interrupt ключ ако има ше бъдат позиционирани в []

================================ Human Message =================================

Plan a one-night business trip to Berlin for Dana. Search options, book a morning flight, and reserve one hotel night for up to 150 EUR. If there are many possible options, take the initiative and select the most affordable one.
================================== Ai Message ==================================
Tool Calls:
  search_travel_options (call_gqRWpSoCHRNU8ov9KdkwJLXa)
 Call ID: call_gqRWpSoCHRNU8ov9KdkwJLXa
  Args:
    destination: Berlin
================================= Tool Message =================================
Name: search_travel_options

{
  "destination": "Berlin",
  "flights": [
    {
      "flight_code": "LH170",
      "departure": "08:10",
      "price_eur": 189
    },
    {
      "flight_code": "FR402",
      "departure": "09:05",
      "price_eur": 129
    }
  ],
  "hotels": [
    {
      "hotel_name": "Asd Hub Hotel",
      "price_eur": 156
    },
    {
      "hotel_name": "Spring V

In [18]:
result_1

{'messages': [HumanMessage(content='Plan a one-night business trip to Berlin for Dana. Search options, book a morning flight, and reserve one hotel night for up to 150 EUR. If there are many possible options, take the initiative and select the most affordable one.', additional_kwargs={}, response_metadata={}, id='2479185c-5bed-45d7-9ddd-06b7708773fd'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 153, 'prompt_tokens': 244, 'total_tokens': 397, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EH7GIGKvA4Gr30jXAAUJCaXdBpXlF', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a03e18-a44

отказваме

In [25]:
result_2 = agent.invoke(
    input = Command(resume = {
        "decisions": [
            {"type": "approve"}, # за първия interrupt Tool: book_flight
            {"type": "approve"}, # за втория interrupt Tool: book_hotel
        ]
    }),
    config = {
        "configurable": {
            "thread_id": "berlin_trip_3"
        }
    }
)

In [26]:
print_conversation(result_2['messages'])

================================ Human Message =================================

Plan a one-night business trip to Berlin for Dana. Search options, book a morning flight, and reserve one hotel night for up to 150 EUR. If there are many possible options, take the initiative and select the most affordable one.
================================== Ai Message ==================================
Tool Calls:
  search_travel_options (call_gqRWpSoCHRNU8ov9KdkwJLXa)
 Call ID: call_gqRWpSoCHRNU8ov9KdkwJLXa
  Args:
    destination: Berlin
================================= Tool Message =================================
Name: search_travel_options

{
  "destination": "Berlin",
  "flights": [
    {
      "flight_code": "LH170",
      "departure": "08:10",
      "price_eur": 189
    },
    {
      "flight_code": "FR402",
      "departure": "09:05",
      "price_eur": 129
    }
  ],
  "hotels": [
    {
      "hotel_name": "Asd Hub Hotel",
      "price_eur": 156
    },
    {
      "hotel_name": "Spring V